employee_payments

In [0]:
%sql
-- employee_payments
DROP TABLE IF EXISTS employee_payments;
CREATE TABLE employee_payments (
emp_id INT PRIMARY KEY,
emp_name VARCHAR(50),
department VARCHAR(30),
base_salary DECIMAL(10,2),
bonus DECIMAL(10,2),
joining_date DATE
);
INSERT INTO employee_payments VALUES
(1,'karthik','Data',75000.75,5000.50,'2019-03-15'),
(2,'veena','HR',65000.40,4000.25,'2021-06-20'),
(3,'ravi','Data',85000.90,6000.75,'2016-01-10'),
(4,'anil','Finance',70000.10,NULL,'2020-09-01'),
(5,'suresh','HR',60000.55,3000.30,'2022-11-25');

























num_affected_rows,num_inserted_rows
5,5


In [0]:
SELECT 
    emp_id,

    -- Case conversions
    UPPER(emp_name) AS upper_name,
    LOWER(emp_name) AS lower_name,
    INITCAP(emp_name) AS proper_name,

    department,

    -- NULL-safe total income calculation
    ROUND(base_salary + COALESCE(bonus, 0)) AS total_income,

    -- Extract joining year
    EXTRACT(YEAR FROM joining_date) AS joining_year,

    -- Experience calculation + classification
    CASE 
        WHEN (EXTRACT(YEAR FROM CURRENT_DATE) - EXTRACT(YEAR FROM joining_date)) > 7 THEN 'Senior'
        WHEN (EXTRACT(YEAR FROM CURRENT_DATE) - EXTRACT(YEAR FROM joining_date)) BETWEEN 4 AND 7 THEN 'Mid'
        ELSE 'Junior'
    END AS employee_level

FROM employee_payments;

emp_id,upper_name,lower_name,proper_name,department,total_income,joining_year,employee_level
1,KARTHIK,karthik,Karthik,Data,80001,2019,Mid
2,VEENA,veena,Veena,HR,69001,2021,Mid
3,RAVI,ravi,Ravi,Data,91002,2016,Senior
4,ANIL,anil,Anil,Finance,70000,2020,Mid
5,SURESH,suresh,Suresh,HR,63001,2022,Mid


orders_delivery

In [0]:
-- orders_delivery
DROP TABLE IF EXISTS orders_delivery;
CREATE TABLE orders_delivery (
order_id INT,
customer_name VARCHAR(50),
order_date DATE,
delivery_date DATE,
order_amount DECIMAL(10,2)
);
INSERT INTO orders_delivery VALUES
(101,'rajesh','2025-01-01','2025-01-05',12500.75),
(102,'meena','2025-01-10','2025-01-10',8400.40),
(103,'arun','2025-01-15','2025-01-20',15600.90),
(104,'pooja','2025-01-18',NULL,9200.10);


In [0]:
SELECT 
    order_id,

    UPPER(customer_name) AS customer_name_upper,

    order_date,

    COALESCE(delivery_date, CURRENT_DATE) AS final_delivery_date,

    DATEDIFF(COALESCE(delivery_date, CURRENT_DATE), order_date) AS delivery_days,

    -- ✅ Correct truncation to 1 decimal
    FLOOR(order_amount * 10) / 10 AS truncated_amount,

    CASE 
        WHEN delivery_date IS NULL THEN 'Pending'
        WHEN DATEDIFF(delivery_date, order_date) = 0 THEN 'Same-day'
        WHEN DATEDIFF(delivery_date, order_date) > 3 THEN 'Delayed'
        ELSE 'On-time'
    END AS delivery_status

FROM orders_delivery;

order_id,customer_name_upper,order_date,final_delivery_date,delivery_days,truncated_amount,delivery_status
101,RAJESH,2025-01-01,2025-01-05,4,12500.700000,Delayed
102,MEENA,2025-01-10,2025-01-10,0,8400.400000,Same-day
103,ARUN,2025-01-15,2025-01-20,5,15600.900000,Delayed
104,POOJA,2025-01-18,2026-04-12,449,9200.100000,Pending


In [0]:
-- customer_spending
DROP TABLE IF EXISTS customer_spending;
CREATE TABLE customer_spending (
cust_id INT,
cust_name VARCHAR(50),
city VARCHAR(30),
purchase_amount DECIMAL(10,2),
purchase_date DATE
);
INSERT INTO customer_spending VALUES
(1,'amit','mumbai',12000.75,'2024-12-01'),
(2,'neha','delhi',8500.40,'2024-12-15'),
(3,'rohit','mumbai',15500.90,'2024-11-20'),
(4,'kavya','chennai',6000.10,'2024-10-05');

In [0]:
SELECT 

    -- First letter capitalized
    INITCAP(cust_name) AS customer_name,

    -- Month name of purchase
    DATE_FORMAT(purchase_date, 'MMMM') AS month_name,

    -- Rounded purchase amount
    ROUND(purchase_amount) AS rounded_amount,

    -- Absolute value (defensive logic)
    ABS(purchase_amount) AS absolute_amount,

    -- Classification
    CASE 
        WHEN purchase_amount > 15000 THEN 'High Spender'
        WHEN purchase_amount BETWEEN 8000 AND 15000 THEN 'Medium Spender'
        ELSE 'Low Spender'
    END AS spender_category

FROM customer_spending;

customer_name,month_name,rounded_amount,absolute_amount,spender_category
Amit,December,12001,12000.75,Medium Spender
Neha,December,8500,8500.40,Medium Spender
Rohit,November,15501,15500.90,High Spender
Kavya,October,6000,6000.10,Low Spender


In [0]:
-- subscriptions
DROP TABLE IF EXISTS subscriptions;
CREATE TABLE subscriptions (
user_id INT,
user_email VARCHAR(100),
start_date DATE,
end_date DATE,
subscription_fee DECIMAL(10,2)
);
INSERT INTO subscriptions VALUES
(1,'karthik@gmail.com','2024-01-01','2025-01-01',12000.50),

(2,'veena@yahoo.com','2024-06-15','2024-12-15',8500.75),

(3,'ravi@hotmail.com','2023-03-01','2024-03-01',15000.90);

In [0]:
SELECT 

    user_id,

    -- Extract username from email
    SPLIT(user_email, '@')[0] AS username,

    start_date,
    end_date,

    -- Replace NULL end_date (if any) with today
    COALESCE(end_date, CURRENT_DATE) AS final_end_date,

    -- Subscription duration (in days)
    DATEDIFF(COALESCE(end_date, CURRENT_DATE), start_date) AS subscription_days,

    -- Rounded fee
    ROUND(subscription_fee) AS rounded_fee,

    -- Status check
    CASE 
        WHEN CURRENT_DATE BETWEEN start_date AND end_date THEN 'Active'
        WHEN CURRENT_DATE > end_date THEN 'Expired'
        ELSE 'Not Started'
    END AS subscription_status

FROM subscriptions;

user_id,username,start_date,end_date,final_end_date,subscription_days,rounded_fee,subscription_status
1,karthik,2024-01-01,2025-01-01,2025-01-01,366,12001,Expired
2,veena,2024-06-15,2024-12-15,2024-12-15,183,8501,Expired
3,ravi,2023-03-01,2024-03-01,2024-03-01,366,15001,Expired


In [0]:
-- loan_details 
DROP TABLE IF EXISTS loan_details;
CREATE TABLE loan_details (
loan_id INT,
customer_name VARCHAR(50),
loan_amount DECIMAL(12,2),
interest_rate DECIMAL(5,2),
loan_start DATE
);
INSERT INTO loan_details VALUES
(201,'suresh',500000.75,8.5,'2022-01-10'),
(202,'mahesh',750000.40,9.2,'2021-05-20'),
(203,'anita',300000.90,7.8,'2023-07-01');

In [0]:
SELECT 

    loan_id,

    -- Uppercase customer name
    UPPER(customer_name) AS customer_name,

    loan_amount,
    interest_rate,

    -- Years since loan start
    FLOOR(DATEDIFF(CURRENT_DATE, loan_start) / 365) AS years_since_start,

    -- Monthly interest rate using POWER
    (POWER(1 + (interest_rate / 100), 1.0/12) - 1) AS monthly_interest_rate,

    -- EMI Calculation (Assume 5-year tenure = 60 months)
    ROUND(
        loan_amount * 
        (POWER(1 + (interest_rate/100)/12, 60) * (interest_rate/100)/12) /
        (POWER(1 + (interest_rate/100)/12, 60) - 1)
    ) AS emi_amount,

    -- Risk Categorization
    CASE 
        WHEN interest_rate > 9 THEN 'High Risk'
        WHEN interest_rate BETWEEN 8 AND 9 THEN 'Medium Risk'
        ELSE 'Low Risk'
    END AS risk_category

FROM loan_details;

loan_id,customer_name,loan_amount,interest_rate,years_since_start,monthly_interest_rate,emi_amount,risk_category
201,SURESH,500000.75,8.50,4,0.006821465987134401,10258.0,Medium Risk
202,MAHESH,750000.40,9.20,4,0.007361171634041375,15642.0,High Risk
203,ANITA,300000.90,7.80,2,0.00627855904221386,6054.0,Low Risk


In [0]:
DROP TABLE IF EXISTS attendance;
CREATE TABLE attendance (
emp_id INT,
emp_name VARCHAR(50),
total_days INT,
present_days INT,
record_date DATE
);
INSERT INTO attendance VALUES
(1,'karthik',30,28,'2025-01-31'),
(2,'veena',30,22,'2025-01-31'),
(3,'ravi',30,18,'2025-01-31');

In [0]:
SELECT 

    emp_id,

    -- Lowercase employee name
    LOWER(emp_name) AS emp_name,

    total_days,
    present_days,

    -- Attendance percentage (rounded)
    ROUND((present_days * 100.0) / total_days) AS attendance_percentage,

    -- Month name
    DATE_FORMAT(record_date, 'MMMM') AS month_name,

    -- Difference (absent days)
    (total_days - present_days) AS absent_days,

    -- Classification
    CASE 
        WHEN (present_days * 100.0) / total_days >= 90 THEN 'Excellent'
        WHEN (present_days * 100.0) / total_days BETWEEN 75 AND 89 THEN 'Average'
        ELSE 'Poor'
    END AS performance

FROM attendance;

emp_id,emp_name,total_days,present_days,attendance_percentage,month_name,absent_days,performance
1,karthik,30,28,93,January,2,Excellent
2,veena,30,22,73,January,8,Poor
3,ravi,30,18,60,January,12,Poor


In [0]:
DROP TABLE IF EXISTS product_sales;
CREATE TABLE product_sales (
product_id INT,
product_name VARCHAR(50),
mrp DECIMAL(10,2),
selling_price DECIMAL(10,2),
sale_date DATE
);
INSERT INTO product_sales VALUES
(1,'Laptop',75000.75,68000.50,'2025-01-10'),
(2,'Mobile',35000.40,33000.25,'2025-01-12'),
(3,'Tablet',25000.90,26000.75,'2025-01-15');

In [0]:
SELECT 

    emp_id,

    LOWER(emp_name) AS emp_name,

    total_days,
    present_days,

    -- Calculate once
    ROUND((present_days * 100.0) / total_days) AS attendance_percentage,

    DATE_FORMAT(record_date, 'MMMM') AS month_name,

    (total_days - present_days) AS absent_days,

    -- Use same rounded value for classification
    CASE 
        WHEN ROUND((present_days * 100.0) / total_days) >= 90 THEN 'Excellent'
        WHEN ROUND((present_days * 100.0) / total_days) BETWEEN 75 AND 89 THEN 'Average'
        ELSE 'Poor'
    END AS performance

FROM attendance;

emp_id,emp_name,total_days,present_days,attendance_percentage,month_name,absent_days,performance
1,karthik,30,28,93,January,2,Excellent
2,veena,30,22,73,January,8,Poor
3,ravi,30,18,60,January,12,Poor


In [0]:
drop table if exists insurance_policies;
CREATE TABLE insurance_policies (
policy_id INT,
holder_name VARCHAR(50),
premium_amount DECIMAL(10,2),
policy_start DATE,
policy_end DATE
);
INSERT INTO insurance_policies VALUES

(301,'arjun',12000.50,'2023-01-01','2026-01-01'),

(302,'megha',8500.75,'2022-06-15','2025-06-15'),

(303,'vinod',15000.90,'2021-03-01','2024-03-01');

num_affected_rows,num_inserted_rows
3,3


In [0]:
SELECT 

    policy_id,

    -- Uppercase holder name
    UPPER(holder_name) AS holder_name,

    premium_amount,

    policy_start,
    policy_end,

    -- Policy age (how long policy has existed)
    FLOOR(DATEDIFF(CURRENT_DATE, policy_start) / 365) AS policy_age_years,

    -- Remaining days
    DATEDIFF(policy_end, CURRENT_DATE) AS remaining_days,

    -- Aging bucket classification
    CASE 
        WHEN policy_end < CURRENT_DATE THEN 'Expired'
        WHEN DATEDIFF(policy_end, CURRENT_DATE) <= 30 THEN 'Expiring Soon'
        WHEN DATEDIFF(policy_end, CURRENT_DATE) BETWEEN 31 AND 365 THEN 'Active'
        ELSE 'Long Active'
    END AS policy_status

FROM insurance_policies;

policy_id,holder_name,premium_amount,policy_start,policy_end,policy_age_years,remaining_days,policy_status
301,ARJUN,12000.50,2023-01-01,2026-01-01,3,-101,Expired
302,MEGHA,8500.75,2022-06-15,2025-06-15,3,-301,Expired
303,VINOD,15000.90,2021-03-01,2024-03-01,5,-772,Expired


In [0]:
drop table if exists salary_revision;
CREATE TABLE salary_revision (

emp_id INT,

emp_name VARCHAR(50),

current_salary DECIMAL(10,2),

rating INT,

last_hike DATE);
INSERT INTO salary_revision VALUES
(1,'karthik',75000.75,5,'2023-01-01'),
(2,'veena',65000.40,4,'2024-01-01'),
(3,'ravi',85000.90,3,'2022-01-01');

num_affected_rows,num_inserted_rows
3,3


In [0]:
SELECT 

    emp_id,

    -- Lowercase employee name
    LOWER(emp_name) AS emp_name,

    current_salary,
    rating,

    -- Years since last hike
    FLOOR(DATEDIFF(CURRENT_DATE, last_hike) / 365) AS years_since_hike,

    -- Increment based on rating
    CASE 
        WHEN rating = 5 THEN current_salary * 0.20
        WHEN rating = 4 THEN current_salary * 0.10
        WHEN rating = 3 THEN current_salary * 0.05
        ELSE 0
    END AS increment_amount,

    -- New salary (rounded)
    ROUND(
        current_salary + 
        CASE 
            WHEN rating = 5 THEN current_salary * 0.20
            WHEN rating = 4 THEN current_salary * 0.10
            WHEN rating = 3 THEN current_salary * 0.05
            ELSE 0
        END
    ) AS new_salary,

    -- Classification
    CASE 
        WHEN rating = 5 THEN 'High Increment'
        WHEN rating = 4 THEN 'Moderate'
        ELSE 'No Increment'
    END AS increment_category

FROM salary_revision;

emp_id,emp_name,current_salary,rating,years_since_hike,increment_amount,new_salary,increment_category
1,karthik,75000.75,5,3,15000.1500,90001,High Increment
2,veena,65000.40,4,2,6500.0400,71500,Moderate
3,ravi,85000.90,3,4,4250.0450,89251,No Increment


In [0]:
drop table if exists bank_accounts;
CREATE TABLE bank_accounts (
account_id INT,
customer_name VARCHAR(50),
balance DECIMAL(12,2),
last_transaction DATE,
branch VARCHAR(30)
);

INSERT INTO bank_accounts VALUES
(501,'ramesh',125000.75,'2024-12-20','hyderabad'),
(502,'sita',8500.40,'2023-06-15','delhi'),
(503,'manoj',-2500.90,'2025-01-05','mumbai');


num_affected_rows,num_inserted_rows
3,3


In [0]:
SELECT 

    account_id,

    customer_name,

    -- Absolute balance
    ABS(balance) AS absolute_balance,

    balance,

    -- Days since last transaction
    DATEDIFF(CURRENT_DATE, last_transaction) AS days_since_last_txn,

    -- Proper case branch name
    INITCAP(branch) AS branch_name,

    -- Sign of balance
    CASE 
        WHEN balance > 0 THEN 'Positive'
        WHEN balance < 0 THEN 'Negative'
        ELSE 'Zero'
    END AS balance_sign,

    -- Account status classification
    CASE 
        WHEN balance < 0 THEN 'Overdrawn'
        WHEN DATEDIFF(CURRENT_DATE, last_transaction) > 365 THEN 'Dormant'
        ELSE 'Active'
    END AS account_status

FROM bank_accounts;

account_id,customer_name,absolute_balance,balance,days_since_last_txn,branch_name,balance_sign,account_status
501,ramesh,125000.75,125000.75,478,Hyderabad,Positive,Dormant
502,sita,8500.40,8500.40,1032,Delhi,Positive,Dormant
503,manoj,2500.90,-2500.90,462,Mumbai,Negative,Overdrawn
